# Predicting war events

## I. Importing essential libraries

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import os
from pathlib import Path
import json

## II. Importing

In [2]:
df_war_events_raw = pd.read_csv("../data/alarms-240222-010325.csv", sep=";")
df_regions = pd.read_csv("../data/regions.csv")

In [3]:
json_path_isw = Path("../data/isw_reports.json")

with open(json_path_isw, "r", encoding="utf-8") as f:
    data = json.load(f)

df_isw_raw = pd.DataFrame(data)

print(type(df_isw_raw))

<class 'pandas.core.frame.DataFrame'>


In [4]:
json_path_tg = Path("../data/telegram_data.json")

with open(json_path_tg, "r", encoding="utf-8") as f:
    data = json.load(f)

df_tg_raw = pd.DataFrame(data)

print(type(df_tg_raw))

<class 'pandas.core.frame.DataFrame'>


## III. EDA

### War events (alarms)

In [5]:
df_war_events_raw.shape

(55788, 6)

In [6]:
df_war_events_raw.head()

,id,region_id,region_city,all_region,start,end
0,52432,12,Львівська обл.,1,2022-02-24 07:43:17,2022-02-24 09:52:28
1,53292,23,Чернігівська обл.,1,2022-02-24 14:00:43,2022-02-24 17:11:43
2,52080,3,Вінницька обл.,1,2022-02-24 15:40:42,2022-02-24 16:10:42
3,52857,19,Харківська обл.,1,2022-02-24 20:11:47,2022-02-24 20:59:47
4,52700,18,Тернопільська обл.,1,2022-02-25 01:59:36,2022-02-25 09:00:19


In [7]:
df_war_events_raw.sample(5)

,id,region_id,region_city,all_region,start,end
7702,68212,19,Харківська обл.,1,2022-07-23 14:11:20,2022-07-23 14:12:37
15255,69008,19,Харківська обл.,1,2023-01-24 20:58:55,2023-01-24 21:30:44
746,2888,9,Київ,0,2022-03-11 11:00:04,2022-03-11 12:30:38
6791,68090,19,Харківська обл.,1,2022-06-27 12:55:44,2022-06-27 13:09:30
51432,153691,20,Херсонська обл.,1,2024-12-06 19:24:11,2024-12-06 20:12:56


In [8]:
df_war_events_raw.describe()

,id,region_id,all_region
count,55788.000000,55788.000000,55788.000000
mean,109103.029935,12.178121,0.972180
std,38574.559928,6.474089,0.164457
min,1.000000,1.000000,0.000000
25%,68259.750000,6.000000,1.000000
50%,126918.500000,13.000000,1.000000
75%,143399.250000,19.000000,1.000000
max,158665.000000,25.000000,1.000000


In [9]:
df_war_events_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 55788 entries, 0 to 55787
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   id           55788 non-null  int64 
 1   region_id    55788 non-null  int64 
 2   region_city  55788 non-null  object
 3   all_region   55788 non-null  int64 
 4   start        55788 non-null  object
 5   end          55788 non-null  object
dtypes: int64(3), object(3)
memory usage: 2.6+ MB


In [10]:
print("Missing values:\n", df_war_events_raw.isna().sum())
print("\nDuplicate rows:", df_war_events_raw.duplicated().sum())
print("Duplicate id:", df_war_events_raw.duplicated(subset=["id"]).sum())

Missing values:
 id             0
region_id      0
region_city    0
all_region     0
start          0
end            0
dtype: int64

Duplicate rows: 0
Duplicate id: 0


In [11]:
df_war_events = df_war_events_raw.copy()

In [12]:
df_war_events["start"] = pd.to_datetime(df_war_events["start"], errors="coerce")
df_war_events["end"] = pd.to_datetime(df_war_events["end"], errors="coerce")

print("Invalid start dates:", df_war_events["start"].isna().sum())
print("Invalid end dates:", df_war_events["end"].isna().sum())

print(df_war_events[["start", "end"]].dtypes)

Invalid start dates: 0
Invalid end dates: 0
start    datetime64[ns]
end      datetime64[ns]
dtype: object


In [13]:
print("Min start:", df_war_events["start"].min())
print("Max start:", df_war_events["start"].max())

print("Min end:", df_war_events["end"].min())
print("Max end:", df_war_events["end"].max())

Min start: 2022-02-24 07:43:17
Max start: 2025-03-01 23:26:07
Min end: 2022-02-24 09:52:28
Max end: 2025-03-02 02:44:07


In [14]:
df_war_events["duration_min"] = (df_war_events["end"] - df_war_events["start"]).dt.total_seconds() / 60

print(df_war_events["duration_min"].describe())

count    55788.000000
mean        72.798103
std         93.094316
min       -781.700000
25%         26.566667
50%         39.733333
75%         84.716667
max       3031.300000
Name: duration_min, dtype: float64


In [15]:
print("\nNegative durations:", (df_war_events["duration_min"] < 0).sum())
print("Zero durations:", (df_war_events["duration_min"] == 0).sum())


Negative durations: 1
Zero durations: 0


In [16]:
df_war_events[df_war_events["duration_min"] < 0]

,id,region_id,region_city,all_region,start,end,duration_min
47970,150000,17,Сумська обл.,1,2024-10-01 20:53:04,2024-10-01 07:51:22,-781.7


One anomalous record with a **negative duration** was found

In [17]:
df_war_events["region_key"] = (
    df_war_events["region_city"]
    .str.replace(" обл.", "")
    .replace({"Крим": "АР Крим"})
)

In [18]:
df_war_events.head()

,id,region_id,region_city,all_region,start,end,duration_min,region_key
0,52432,12,Львівська обл.,1,2022-02-24 07:43:17,2022-02-24 09:52:28,129.183333,Львівська
1,53292,23,Чернігівська обл.,1,2022-02-24 14:00:43,2022-02-24 17:11:43,191.000000,Чернігівська
2,52080,3,Вінницька обл.,1,2022-02-24 15:40:42,2022-02-24 16:10:42,30.000000,Вінницька
3,52857,19,Харківська обл.,1,2022-02-24 20:11:47,2022-02-24 20:59:47,48.000000,Харківська
4,52700,18,Тернопільська обл.,1,2022-02-25 01:59:36,2022-02-25 09:00:19,420.716667,Тернопільська


In [19]:
alarm_regions = sorted(
    df_war_events[df_war_events["all_region"] == 1]["region_key"].unique()
)

regions = df_regions["region"]

missing_in_alarms = sorted(set(regions) - set(alarm_regions))
extra_in_alarms = sorted(set(alarm_regions) - set(regions))

print("Missing regions in alarms:", missing_in_alarms)
print("Extra regions in alarms:", extra_in_alarms)

Missing regions in alarms: ['Луганська']
Extra regions in alarms: []


**Luhansk region is missing** in the alarms dataset.

In [20]:
alarm_id_check = (
    df_war_events[df_war_events["all_region"] == 1][["region_id", "region_key"]]
    .drop_duplicates()
    .rename(columns={"region_id": "region_id_alarms"})
)

regions_id_check = (
    df_regions[["region_id", "region"]]
    .rename(columns={"region_id": "region_id_regions", "region": "region_key"})
)

id_check = alarm_id_check.merge(regions_id_check, on="region_key")

id_mismatch = id_check[id_check["region_id_alarms"] != id_check["region_id_regions"]]

print("Number of ID mismatches:", len(id_mismatch))
id_mismatch

Number of ID mismatches: 19


,region_id_alarms,region_key,region_id_regions
0,12,Львівська,13
1,23,Чернігівська,25
2,3,Вінницька,2
3,19,Харківська,20
4,18,Тернопільська,19
5,16,Рівненська,17
6,22,Черкаська,23
7,14,Одеська,15
9,2,Волинська,3
11,20,Херсонська,21


### ISW

In [21]:
df_isw_raw.shape

(1439, 4)

In [22]:
df_isw_raw.head()

,date,title,url,text
0,2022-02-25,Russia-Ukraine Warning Update: Russian Offensi...,https://understandingwar.org/research/russia-u...,Previous\nNext\nRussia-Ukraine Warning Update:...
1,2022-02-26,Russia-Ukraine Warning Update: Russian Offensi...,https://understandingwar.org/research/russia-u...,Previous\nNext\nRussia-Ukraine Warning Update:...
2,2022-02-27,Russia-Ukraine Warning Update: Russian Offensi...,https://understandingwar.org/research/russia-u...,Previous\nNext\nRussia-Ukraine Warning Update:...
3,2022-02-28,"Russian Offensive Campaign Assessment, Februar...",https://understandingwar.org/research/russia-u...,Previous\nNext\nRussian Offensive Campaign Ass...
4,2022-03-01,"Russian Offensive Campaign Assessment, March 1...",https://understandingwar.org/research/russia-u...,Previous\nNext\nRussian Offensive Campaign Ass...


In [23]:
df_isw_raw.sample(5)

,date,title,url,text
367,2023-03-10,"Russian Offensive Campaign Assessment, March 1...",https://understandingwar.org/research/russia-u...,Previous\nNext\nRussian Offensive Campaign Ass...
1430,2026-02-21,"Russian Offensive Campaign Assessment, Februar...",https://understandingwar.org/research/russia-u...,Previous\nNext\nRussian Offensive Campaign Ass...
1401,2026-01-23,"Russian Offensive Campaign Assessment, January...",https://understandingwar.org/research/russia-u...,Previous\nNext\nRussian Offensive Campaign Ass...
579,2023-10-11,"Russian Offensive Campaign Assessment, October...",https://understandingwar.org/research/russia-u...,Previous\nNext\nRussian Offensive Campaign Ass...
280,2022-12-11,"Russian Offensive Campaign Assessment, Decembe...",https://understandingwar.org/research/russia-u...,Previous\nNext\nRussian Offensive Campaign Ass...


In [24]:
df_isw_raw.describe()

,date,title,url,text
count,1439,1439,1439,1439
unique,1439,1439,1439,1439
top,2026-03-01,"Russian Offensive Campaign Assessment, March 1...",https://understandingwar.org/research/russia-u...,Previous\nNext\nRussian Offensive Campaign Ass...
freq,1,1,1,1


In [25]:
df_isw_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1439 entries, 0 to 1438
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   date    1439 non-null   object
 1   title   1439 non-null   object
 2   url     1439 non-null   object
 3   text    1439 non-null   object
dtypes: object(4)
memory usage: 45.1+ KB


All values are unique.  
All columns currently have the `object` data type. 

In [26]:
df_isw = df_isw_raw.copy()

In [27]:
df_isw["date"] = pd.to_datetime(df_isw["date"], errors="coerce")
print("Invalid dates:", df_isw["date"].isna().sum())
print(df_isw["date"].dtype)

Invalid dates: 0
datetime64[ns]


In [28]:
print("Duplicate dates:", df_isw["date"].duplicated().sum())

Duplicate dates: 0


In [29]:
all_dates = pd.date_range("2022-02-24", "2026-03-01")
existing_dates = pd.DatetimeIndex(df_isw["date"])
missing_dates = all_dates.difference(existing_dates)

print("Missing dates:", len(missing_dates))
print(missing_dates)

Missing dates: 28
DatetimeIndex(['2022-02-24', '2022-03-16', '2022-03-18', '2022-03-29',
               '2022-04-01', '2022-04-11', '2022-05-14', '2022-06-23',
               '2022-10-28', '2022-11-24', '2022-12-25', '2023-01-01',
               '2023-03-19', '2023-07-08', '2023-07-27', '2023-11-23',
               '2023-12-25', '2024-01-01', '2024-01-05', '2024-10-08',
               '2024-11-28', '2024-12-25', '2025-01-01', '2025-09-07',
               '2025-10-31', '2025-11-27', '2025-12-25', '2026-01-01'],
              dtype='datetime64[ns]', freq=None)


In [30]:
outside_range = df_isw[(df_isw["date"] < "2022-02-24") | (df_isw["date"] > "2026-03-01")]
print(outside_range)

Empty DataFrame
Columns: [date, title, url, text]
Index: []


In [31]:
def clean_isw_text(text):
    text = str(text)
    text = text.replace("Previous\nNext", "")
    text = text.replace("\n", " ")
    return text

df_isw["text_clean"] = df_isw["text"].apply(clean_isw_text)

In [32]:
df_isw.head()

,date,title,url,text,text_clean
0,2022-02-25,Russia-Ukraine Warning Update: Russian Offensi...,https://understandingwar.org/research/russia-u...,Previous\nNext\nRussia-Ukraine Warning Update:...,Russia-Ukraine Warning Update: Russian Offens...
1,2022-02-26,Russia-Ukraine Warning Update: Russian Offensi...,https://understandingwar.org/research/russia-u...,Previous\nNext\nRussia-Ukraine Warning Update:...,Russia-Ukraine Warning Update: Russian Offens...
2,2022-02-27,Russia-Ukraine Warning Update: Russian Offensi...,https://understandingwar.org/research/russia-u...,Previous\nNext\nRussia-Ukraine Warning Update:...,Russia-Ukraine Warning Update: Russian Offens...
3,2022-02-28,"Russian Offensive Campaign Assessment, Februar...",https://understandingwar.org/research/russia-u...,Previous\nNext\nRussian Offensive Campaign Ass...,"Russian Offensive Campaign Assessment, Februa..."
4,2022-03-01,"Russian Offensive Campaign Assessment, March 1...",https://understandingwar.org/research/russia-u...,Previous\nNext\nRussian Offensive Campaign Ass...,"Russian Offensive Campaign Assessment, March ..."


In [33]:
df_isw["text"] = df_isw["text_clean"]
df_isw = df_isw.drop(columns=["text_clean"])

In [34]:
df_isw.head()

,date,title,url,text
0,2022-02-25,Russia-Ukraine Warning Update: Russian Offensi...,https://understandingwar.org/research/russia-u...,Russia-Ukraine Warning Update: Russian Offens...
1,2022-02-26,Russia-Ukraine Warning Update: Russian Offensi...,https://understandingwar.org/research/russia-u...,Russia-Ukraine Warning Update: Russian Offens...
2,2022-02-27,Russia-Ukraine Warning Update: Russian Offensi...,https://understandingwar.org/research/russia-u...,Russia-Ukraine Warning Update: Russian Offens...
3,2022-02-28,"Russian Offensive Campaign Assessment, Februar...",https://understandingwar.org/research/russia-u...,"Russian Offensive Campaign Assessment, Februa..."
4,2022-03-01,"Russian Offensive Campaign Assessment, March 1...",https://understandingwar.org/research/russia-u...,"Russian Offensive Campaign Assessment, March ..."


### Telegram 

In [35]:
df_tg_raw.shape

(129477, 3)

In [36]:
df_tg_raw.head()

,date,channel,message
0,2026-03-06 11:06:58+00:00,DeepStateUA,**🔄**** Мапу оновлено\n**\n⚔️ Ворог просунувся...
1,2026-03-06 10:57:29+00:00,DeepStateUA,**🇺🇦**** Другий день обміну: додому **[**повер...
2,2026-03-06 06:32:47+00:00,DeepStateUA,🤬 **Угорщина затримала інкасаторські автомобіл...
3,2026-03-05 14:46:58+00:00,DeepStateUA,**🔄**** Мапу оновлено\n**\n⚔️ Ворог просунувся...
4,2026-03-05 13:46:39+00:00,DeepStateUA,📋 **Зачистка з боку Сил Оборони на стику Запор...


In [37]:
df_tg_raw.sample(5)

,date,channel,message
40731,2023-12-31 15:14:20+00:00,UkraineNow,**Журналісти німецького телеканалу ZDF потрапи...
6397,2023-03-13 23:50:52+00:00,DeepStateUA,🕶🕶🕶 **на фронтах станом на 13 березня 2023 рок...
72418,2022-03-02 08:23:49+00:00,UkraineNow,‼️Львів тривога !!!
107551,2024-11-11 10:11:51+00:00,kpszsu,📢Відбій загрози застосування балістичного озбр...
65654,2022-05-07 14:33:38+00:00,UkraineNow,❗️ **Пожежу на площі близько 900 кв.м ліквідов...


In [38]:
df_tg_raw.describe()

,date,channel,message
count,129477,129477,129477
unique,129333,3,107432
top,2025-10-23 03:28:04+00:00,UkraineNow,📢 Відбій загрози.
freq,6,61363,929


In [39]:
df_tg_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 129477 entries, 0 to 129476
Data columns (total 3 columns):
 #   Column   Non-Null Count   Dtype 
---  ------   --------------   ----- 
 0   date     129477 non-null  object
 1   channel  129477 non-null  object
 2   message  129477 non-null  object
dtypes: object(3)
memory usage: 3.0+ MB


In [40]:
print("Unique channels:", df_tg_raw["channel"].nunique())
print(df_tg_raw["channel"].value_counts())

Unique channels: 3
channel
UkraineNow     61363
kpszsu         56272
DeepStateUA    11842
Name: count, dtype: int64


In [41]:
print("Duplicate rows:", df_tg_raw.duplicated().sum())

Duplicate rows: 0


In [42]:
df_tg = df_tg_raw.copy()

In [43]:
df_tg["date"] = pd.to_datetime(df_tg["date"], utc=True, errors="coerce")
print("Invalid dates:", df_tg["date"].isna().sum())
print(df_isw["date"].dtype)

Invalid dates: 0
datetime64[ns]


In [44]:
print("Min date:", df_tg["date"].min())
print("Max date:", df_tg["date"].max())

Min date: 2022-02-24 03:51:12+00:00
Max date: 2026-03-06 16:49:21+00:00


In [45]:
df_tg["message"] = df_tg["message"].astype(str)

print("Empty messages:", (df_tg["message"] == "").sum())
print("Whitespace messages:", df_tg["message"].str.strip().eq("").sum())

print("\nShortest messages:")
print(df_tg.loc[df_tg["message"].str.len().nsmallest(10).index, ["channel", "date", "message"]])

Empty messages: 0
Whitespace messages: 0

Shortest messages:
           channel                      date message
1218   DeepStateUA 2025-06-01 11:49:49+00:00       🕸
4052   DeepStateUA 2024-02-16 12:20:06+00:00       🥪
7472   DeepStateUA 2022-11-11 11:10:16+00:00       🍉
9531   DeepStateUA 2022-04-12 18:46:52+00:00       😎
9804   DeepStateUA 2022-04-01 04:39:05+00:00       😳
10044  DeepStateUA 2022-03-22 16:42:05+00:00       💸
10163  DeepStateUA 2022-03-17 21:02:54+00:00       😐
10670  DeepStateUA 2022-03-04 16:47:06+00:00       🤨
10683  DeepStateUA 2022-03-04 12:53:13+00:00       🍾
10798  DeepStateUA 2022-03-03 07:02:20+00:00       🔴


## IV. Prepare data